# 07 — Difference-in-Differences & Causal Inference

This notebook performs causal inference analysis on organic waste ban policies using:
- **Parallel Trends** visualization to validate DiD assumptions
- **Two-Way Fixed Effects (TWFE) DiD** estimation
- **Event Study** for dynamic treatment effects
- **Synthetic Control** for California SB 1383

Outputs: `parallel_trends.png`, `event_study.png`, `synthetic_california.png`, `did_results.json`, `event_study_results.json`, `synthetic_control_results.json`

---
## 1. Setup & Build Panel

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import pandas as pd, numpy as np, json, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from utils.causal_inference import (
    build_state_year_panel, run_twfe_did,
    run_event_study, run_synthetic_control
)

BAN_COLOR, NOBAN_COLOR = '#4A6B6F', '#B8B8B8'

os.makedirs('outputs/causal/charts', exist_ok=True)

In [3]:
state_year, policy = build_state_year_panel()
print(f"Panel shape: {state_year.shape}")
print(f"Policy states: {len(policy)}")
state_year.head(3)

Panel shape: (750, 18)
Policy states: 12


,state,year,tons_surplus,tons_waste,tons_landfill,tons_composting,tons_donations,tons_anaerobic_digestion,tons_animal_feed,us_dollars_surplus,co2e,water,meals_wasted,landfill_rate,diversion_rate,ban_year,ban_active,is_ban_state
0,Alabama,2010,605567.602551,526539.336201,307629.284782,101596.317651,14262.997676,2945.070690,40430.831922,3.317411e+09,2.722035e+06,1.992007e+11,9.855077e+08,0.584247,0.302418,9999,0,False
1,Alabama,2011,625989.926059,542326.715942,316125.547806,105286.414382,15274.325194,3160.703920,42921.350342,3.432795e+09,2.815341e+06,2.058467e+11,1.017859e+09,0.582906,0.307274,9999,0,False
2,Alabama,2012,643898.998395,557263.486746,325597.048581,107782.742068,16205.356765,3334.223213,43874.216529,3.566739e+09,2.903121e+06,2.134882e+11,1.046156e+09,0.584278,0.307209,9999,0,False


---
## 2. Parallel Trends Visualization

The key identifying assumption for DiD is that treated and control groups
would have followed parallel trends absent treatment. We visualize average
landfill rates for ban vs. non-ban states over time.

In [4]:
treated = state_year[state_year['is_ban_state']].groupby('year')['landfill_rate'].mean()
control = state_year[~state_year['is_ban_state']].groupby('year')['landfill_rate'].mean()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(treated.index, treated.values, 'o-', color=BAN_COLOR, lw=2, label='Ban States (n=12)')
ax.plot(control.index, control.values, 's-', color=NOBAN_COLOR, lw=2, label='Non-Ban (n=38)')
ax.axvline(x=2014, color='red', ls='--', alpha=0.5, label='First bans (2014)')
ax.axvline(x=2022, color='red', ls='--', alpha=0.3, label='CA SB 1383')
ax.set_xlabel('Year')
ax.set_ylabel('Landfill Rate')
ax.set_title('Parallel Trends: Ban vs Non-Ban States')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/parallel_trends.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved parallel_trends.png')

Saved parallel_trends.png


---
## 3. TWFE DiD Estimation

Two-Way Fixed Effects regression with state and year fixed effects.
The coefficient on `post_ban` captures the average treatment effect
of organic waste ban policies on landfill outcomes.

In [5]:
did = run_twfe_did(state_year)
for k, v in did.items():
    print(f"{k}: beta={v['beta']:.6f}, p={v['pvalue']:.4f}")

landfill_rate: beta=0.020266, p=0.0077
diversion_rate: beta=-0.051783, p=0.0000
tons_landfill: beta=30690.232119, p=0.4558


In [6]:
with open('outputs/causal/did_results.json', 'w') as f:
    json.dump({'twfe_did': did}, f, indent=2)
print('Saved did_results.json')

Saved did_results.json


---
## 4. Event Study

The event study decomposes the treatment effect by time relative to
ban implementation. Pre-treatment coefficients near zero validate
the parallel trends assumption; post-treatment coefficients show
the dynamic effect over time.

In [7]:
es = run_event_study(state_year)

fig, ax = plt.subplots(figsize=(10, 6))
t = [r['event_time'] for r in es]
c = [r['coef'] for r in es]
ax.plot(t, c, 'o-', color=BAN_COLOR, lw=2, zorder=3)
ax.fill_between(t, [r['ci_lo'] for r in es], [r['ci_hi'] for r in es],
                alpha=0.2, color=BAN_COLOR)
ax.axhline(y=0, color='black', ls='-', lw=0.5)
ax.axvline(x=-0.5, color='red', ls='--', alpha=0.5, label='Ban Implementation')
ax.set_xlabel('Years Relative to Ban')
ax.set_ylabel('Effect on Landfill Rate')
ax.set_title('Event Study: Dynamic Effects of Organic Waste Bans')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/event_study.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved event_study.png')

Saved event_study.png


In [8]:
with open('outputs/causal/event_study_results.json', 'w') as f:
    json.dump(es, f, indent=2)
print('Saved event_study_results.json')

Saved event_study_results.json


---
## 5. Synthetic Control: California

Synthetic control constructs a weighted combination of non-ban states
that best matches California's pre-treatment trajectory. The gap between
actual and synthetic California after SB 1383 (2022) measures the
causal effect of the policy.

In [9]:
sc = run_synthetic_control(state_year, 'California', 2022, policy)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sc['years'], sc['actual'], 'o-', color=BAN_COLOR, lw=2, label='California (Actual)')
ax.plot(sc['years'], sc['synthetic'], 's--', color=NOBAN_COLOR, lw=2, label='Synthetic California')
ax.axvline(x=2022, color='red', ls='--', alpha=0.5, label='SB 1383 Effective')
ax.set_xlabel('Year')
ax.set_ylabel('Landfill Rate')
ax.set_title('Synthetic Control: California SB 1383 Impact')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/causal/charts/synthetic_california.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved synthetic_california.png')

Saved synthetic_california.png


---
## 6. Save Results

In [10]:
# Convert numpy arrays to lists for JSON serialization
sc_save = {
    'actual': [float(x) for x in sc['actual']],
    'synthetic': [float(x) for x in sc['synthetic']],
    'years': [int(x) for x in sc['years']],
    'weights': {k: float(v) for k, v in sc['weights'].items()},
    'treatment_year': sc['treatment_year']
}
with open('outputs/causal/synthetic_control_results.json', 'w') as f:
    json.dump(sc_save, f, indent=2)
print('Saved synthetic_control_results.json')

Saved synthetic_control_results.json


In [11]:
print('All causal charts saved.')

All causal charts saved.
